In [28]:
from pathlib import Path
import json

# Find the root directory, no matter where we are. 
def find_project_root(marker="README.md"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root()

path = PROJECT_ROOT / "data/raw/api_progress.json"
print(path)

/home/seancm/code/salary_prediction_project/data/raw/api_progress.json


In [29]:
# read all data from path into all_data
with path.open("r") as f:
    all_data = json.load(f)

In [30]:
import pandas as pd

df = pd.json_normalize(all_data)

In [31]:
df.head()

,redirect_url,title,longitude,contract_time,contract_type,created,adref,description,salary_min,__CLASS__,...,salary_max,search_term,location.display_name,location.__CLASS__,location.area,category.tag,category.__CLASS__,category.label,company.display_name,company.__CLASS__
0,https://www.adzuna.com.au/details/5601487332?u...,Data Scientist,152.928532,full_time,permanent,2026-01-27T13:16:39Z,eyJhbGciOiJIUzI1NiJ9.eyJzIjoib2dmUWlmTUc4UkdZZ...,"We are seeking a Data Scientist for a hybrid, ...",140000,Adzuna::API::Response::Job,...,140000,data scientist,"Brisbane, Brisbane Region",Adzuna::API::Response::Location,"[Australia, Queensland, Brisbane Region, Brisb...",it-jobs,Adzuna::API::Response::Category,IT Jobs,GRIT Talent Consulting,Adzuna::API::Response::Company
1,https://www.adzuna.com.au/details/5591225519?u...,Data Scientist,151.203231,full_time,permanent,2026-01-20T00:15:22Z,eyJhbGciOiJIUzI1NiJ9.eyJzIjoib2dmUWlmTUc4UkdZZ...,Job title: Data Scientist Location: Sydney Per...,150000,Adzuna::API::Response::Job,...,150000,data scientist,"Sydney, Sydney Region",Adzuna::API::Response::Location,"[Australia, New South Wales, Sydney Region, Sy...",it-jobs,Adzuna::API::Response::Category,IT Jobs,Source Point Services Pty Ltd,Adzuna::API::Response::Company
2,https://www.adzuna.com.au/details/5592295440?u...,Data Scientist,151.203231,full_time,permanent,2026-01-21T00:24:32Z,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTU5MjI5NTQ0MCIsI...,Data Scientist Operational Analytics | Machine...,130000,Adzuna::API::Response::Job,...,130000,data scientist,"Sydney, Sydney Region",Adzuna::API::Response::Location,"[Australia, New South Wales, Sydney Region, Sy...",it-jobs,Adzuna::API::Response::Category,IT Jobs,Latitude IT,Adzuna::API::Response::Company
3,https://www.adzuna.com.au/details/5612518226?u...,Data Scientist,NaN,full_time,permanent,2026-02-04T10:01:34Z,eyJhbGciOiJIUzI1NiJ9.eyJzIjoib2dmUWlmTUc4UkdZZ...,Get recognition and reward working with this a...,150000,Adzuna::API::Response::Job,...,175000,data scientist,"Sydney, Sydney Region",Adzuna::API::Response::Location,"[Australia, New South Wales, Sydney Region, Sy...",it-jobs,Adzuna::API::Response::Category,IT Jobs,Mane Consulting,Adzuna::API::Response::Company
4,https://www.adzuna.com.au/details/5619738382?u...,Data Scientist,NaN,full_time,contract,2026-02-08T08:11:03Z,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTYxOTczODM4MiIsI...,Department for Health and Wellbeing – Clinical...,121107,Adzuna::API::Response::Job,...,125679,data scientist,Australia,Adzuna::API::Response::Location,[Australia],it-jobs,Adzuna::API::Response::Category,IT Jobs,SA Health,Adzuna::API::Response::Company


In [32]:
# keep redirect column - we want it after filtering to give to our scraper
df = df.drop(columns=["adref"])
# Drop any column that contains __CLASS__
df = df.drop(columns=[col for col in df.columns if "__CLASS__" in col])

In [33]:

df['salary_is_predicted'].value_counts()

salary_is_predicted
0    14765
Name: count, dtype: int64

In [34]:
[len(x) for x in df['location.area']]
df['location.area_length'] = df['location.area'].apply(len)
df['location.area_length'].value_counts()

# Subset to location length == 5 and pull location.area
# df[df['location.area_length'] == 3]['location.area'].value_counts()


location.area_length
5    6116
4    4765
1    1447
6    1086
3     859
2     492
Name: count, dtype: int64

In [35]:
# create location_1, location_2, location_3 by splitting location.area lists into components, 
# accounting for the fact that not all lists have all 5 components
location_cols = ['location_country', 'location_state', 'location_region', 'location_city', 'location_area', 'location_suburb']
location_df = df['location.area'].apply(lambda x: pd.Series(x + [None] * (6 - len(x))))
location_df.columns = location_cols
df = pd.concat([df, location_df], axis=1)

In [36]:
df.loc[df['location_city'] == 'Western Sydney', 'location_suburb'].value_counts()

location_suburb
Parramatta        93
Blacktown         31
Eastern Creek     24
Bankstown         20
Rosemeadow        18
                  ..
Cabramatta         1
Picton             1
Campsie            1
Pendle Hill        1
North Richmond     1
Name: count, Length: 120, dtype: int64

In [37]:
# df = df.loc[df['salary_min'] > 30000, :]

In [38]:
df[df['salary_min'] == 0]['salary_max'].value_counts()

salary_max
120000    58
130000    50
110000    42
166400    33
2500      32
          ..
98000      1
135200     1
101000     1
13000      1
2          1
Name: count, Length: 151, dtype: int64

In [39]:
both = df[(df['salary_min'] > 0) & (df['salary_max'] > 0)]
ratio = (both['salary_max'] / both['salary_min']).median()
print(f"Typical max/min ratio: {ratio:.2f}")

Typical max/min ratio: 1.12


In [ ]:
df['title'] = df['title'].fillna('')
df['description'] = df['description'].fillna('')
df['full_description'] = df['full_description'].fillna('')
df['missing_long_lat'] = df['longitude'].isna() | df['latitude'].isna()

In [ ]:
# Coutn unique rows defined by title + description + salary
df['unique_id'] = df['title'].str.lower() + '|' + df['description'] + '|' + df['avg_salary'].astype(str)

# Count how many rows we would have if we dropped duplicates based on unique_id
print(f"Original rows: {len(df)}")
print(f"Unique rows: {df['unique_id'].nunique()}")

# Drop rows using unique_id. Keep earliest posting (lowest id)
df = df.sort_values('id').drop_duplicates(subset='unique_id', keep='first').reset_index(drop=True)

In [ ]:
df['imputed_salary_min'] = df.apply(lambda row: row['salary_min'] if row['salary_min'] > 0 else row['salary_max'] / ratio, axis=1)
df['imputed_salary_max'] = df.apply(lambda row: row['salary_max'] if row['salary_max'] > 0 else row['salary_min'] * ratio, axis=1)
df['avg_salary'] = (df['imputed_salary_min'] + df['imputed_salary_max']) / 2

In [ ]:


lower = 50_000
upper = 300_000

df = df[df['avg_salary'] >= lower].copy()
df['avg_salary'] = df['avg_salary'].clip(upper=upper)
company_counts = df['company.display_name'].value_counts()
df['company_listing_count'] = df['company.display_name'].map(company_counts)

min_count = 50  # tune this threshold



In [ ]:

for col in ['location_region', 'location_city']:
    counts = df[col].value_counts()
    # Create new column for the abridged versions of the location 
    df[f'{col}_abridged'] = df[col].where(df[col].map(counts) >= min_count, 'Other')

Original rows: 12590
Unique rows: 11721


In [46]:
df['id'].apply(lambda x: type(x)).value_counts()

id
<class 'str'>    12590
Name: count, dtype: int64

In [47]:
df['id'] = df['id'].astype(int)

In [49]:
# Get mean and count of jobs by location_state, sorted by mean salary
state_salary = df.groupby('location_state')['avg_salary'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print(state_salary)

                                       mean  count
location_state                                    
Australian Capital Territory  131350.126323    336
Western Australia             124223.143112    967
Tasmania                      121910.581481    150
Victoria                      118083.368747   2281
New South Wales               117694.901938   4087
Northern Territory            117531.334559    136
Queensland                    116097.613348   2260
South Australia               108265.310209    382


In [50]:
df['category.label'].value_counts().head(20)

category.label
Healthcare & Nursing Jobs           2529
Trade & Construction Jobs           1045
IT Jobs                              951
Engineering Jobs                     838
Accounting & Finance Jobs            796
Admin Jobs                           734
Hospitality & Catering Jobs          610
Teaching Jobs                        511
Sales Jobs                           504
Logistics & Warehouse Jobs           464
Social work Jobs                     421
Legal Jobs                           370
HR & Recruitment Jobs                272
PR, Advertising & Marketing Jobs     265
Retail Jobs                          241
Consultancy Jobs                     157
Manufacturing Jobs                   140
Customer Services Jobs               139
Property Jobs                        139
Scientific & QA Jobs                 119
Name: count, dtype: int64

In [51]:
df.groupby('category.label')['avg_salary'].agg(['mean', 'count']).sort_values('mean', ascending=False)

,mean,count
category.label,,
IT Jobs,151722.011099,951
"Energy, Oil & Gas Jobs",146853.641975,18
Legal Jobs,141314.446697,370
Healthcare & Nursing Jobs,138641.912702,2529
Trade & Construction Jobs,126421.748857,1045
Charity & Voluntary Jobs,126276.147059,17
Engineering Jobs,126167.081809,838
Consultancy Jobs,125463.970984,157
Accounting & Finance Jobs,117307.968802,796


In [52]:
# Save the feature engineered data to the /data/clean_data folder as a csv
output_path = PROJECT_ROOT / "data/processed/feature_engineered_data.csv"
df.to_csv(output_path, index=False)


In [53]:
# Also save just the redirect URLs in JSON format
df['redirect_url'].to_json(PROJECT_ROOT / "data/processed/redirect_urls.json", orient='records')